# TxGNN (PyTorch Geometric) — Training Walkthrough

This notebook is a **guided, end-to-end training recipe** for the PyG port of TxGNN that lives in
[`pyg_implementation/`](pyg_implementation). It explains *what* every stage does, *why* it exists in
the method, and *where* the corresponding code lives — then runs it.

> **Nothing here has been executed.** The cells are authored and ordered so you can run them top to
> bottom, but the heavy stages (data download, pretraining, fine-tuning) are left un-run on purpose.
> Read [§12 Known rough edges](#12--known-rough-edges-read-before-a-long-run) before launching a long job.

**Source material**
- Code: `pyg_implementation/txgnn/{TxData,TxGNN,TxEval,model,utils}.py`
- Paper: *A foundation model for clinician-centered drug repurposing*, Nature Medicine — local copy in
  [`txgnn_article/main.md`](txgnn_article/main.md)
- Port design notes: [`PYG_REFACTOR.md`](PYG_REFACTOR.md)

---
## 1 · What TxGNN actually is

TxGNN predicts, for a **(drug, disease)** pair, the likelihood of three clinical relationships:
`indication`, `contraindication`, and `off-label use`. It does this **zero-shot** — i.e. for diseases
that have *no* known treatment edges in the training graph — which is the whole point of the paper.

It runs on a heterogeneous biomedical knowledge graph (KG): **10 node types, 29 relation types,
~123.5 K nodes and ~8 M edges** (PrimeKG, Harvard Dataverse). Node types you will see in
`G.node_types`:

| Node type | Meaning |
|---|---|
| `drug` | small molecules / biologics |
| `disease` | disease concepts (MONDO) |
| `gene/protein` | targets, interactors |
| `effect/phenotype` | HPO phenotypes |
| `exposure` | environmental exposures |
| `pathway`, `molecular_function`, `cellular_component`, `biological_process`, `anatomy` | ontology context |

The paper describes four modules. Each maps to a concrete place in this codebase:

| # | Module (paper) | Where it lives here |
|---|---|---|
| 1 | Heterogeneous GNN encoder | `model.HeteroRGCNLayer` / `model.AttHeteroRGCNLayer` |
| 2 | Disease-similarity metric-learning decoder | `model.DistMultPredictor` (the `proto=True` branch) |
| 3 | All-relation pretraining → drug–disease fine-tuning | `TxGNN.pretrain()` / `TxGNN.finetune()` |
| 4 | GraphMask explainability | `model.HeteroRGCN.graphmask_forward` + `TxGNN.train_graphmask()` |

---
## 2 · The architecture, step by step

### 2.1 Encoder — relation-specific message passing (2 layers)

Every node $i$ starts with a Xavier-uniform embedding $\mathbf{x}_i$ (**frozen**, `requires_grad=False` —
see `utils.initialize_node_embedding`). For each layer $l$ and each relation $r$:

$$\mathbf{m}^{(l)}_{r,i} = W^{(l)}_{r} \, \mathbf{h}^{(l-1)}_i
\qquad
\widetilde{\mathbf{m}}^{(l)}_{r,i} = \frac{1}{|\mathcal{N}_r(i)|}\sum_{j \in \mathcal{N}_r(i)} \mathbf{m}^{(l)}_{r,j}
\qquad
\mathbf{h}^{(l)}_i = \sum_{r \in \mathcal{T}_R} \widetilde{\mathbf{m}}^{(l)}_{r,i}$$

That is: **linear transform per relation → mean-aggregate within a relation → sum across relations.**
In `HeteroRGCNLayer.forward` this is exactly

```python
Wh  = self.weight[etype](feat_dict[srctype])          # relation-specific W_r
msg = Wh[src_nodes]                                    # gather along edges
agg = scatter(msg, dst_nodes, reduce='mean', ...)      # mean within relation
out[dsttype] = out[dsttype] + agg                      # sum across relations
```

`torch_scatter.scatter` replaces DGL's `multi_update_all(fn.copy_u + fn.mean, 'sum')`. Between the two
layers there is a `leaky_relu`. Node types that receive no incoming edges are filled with zeros so
downstream indexing never breaks.

> **Note on the residual term.** The paper writes $\mathbf{h}^{(l)}_i = \mathbf{h}^{(l-1)}_i + \sum_r \widetilde{\mathbf{m}}$.
> The reference DGL implementation drops the $\mathbf{h}^{(l-1)}_i$ skip term, and this PyG port
> reproduces the code, not the equation — deliberately, since the goal of the port was bit-parity with DGL.

**`attention=True` (the "TxGAT" ablation)** swaps in `AttHeteroRGCNLayer`, which scores each edge with
$e = \mathrm{LeakyReLU}\!\big(a_r^\top [\,W_r h_u \,\|\, W_{\bar r} h_v\,]\big)$, softmaxes over each
destination node (`torch_geometric.utils.softmax`, replacing `dgl.ops.edge_softmax`), and uses those
weights instead of a plain mean. $\bar r$ is the *reverse* relation, resolved by the naming convention
`rev_<rel>` ⇄ `<rel>`.

### 2.2 Decoder — DistMult

Each of the relation types gets a trainable vector $\mathbf{w}_r$ (`self.w_rels`, one row per canonical
edge type), and

$$p_{i,j,r} = \sigma\big(\textstyle\sum (\mathbf{h}_i \odot \mathbf{w}_r \odot \mathbf{h}_j)\big)$$

`DistMultPredictor._apply_edges` — three lines of tensor indexing that replace DGL's `apply_edges` UDF.

### 2.3 Disease-similarity metric learning (`proto=True`)

This is the mechanism that makes zero-shot work, and it is the most intricate part of the code.

**Step 1 — signature vector.** For every disease with at least one treatment edge, build a *binary
neighbourhood profile* $\mathbf{p}_i$ over one-hop neighbours (`utils.obtain_disease_profile`). With the
default `sim_measure='all_nodes_profile'` the profile concatenates indicator vectors over
`disease_disease` and `rev_disease_protein` neighbours. Cosine similarity between profiles
(`utils.sim_matrix`) gives a dense disease–disease similarity matrix, **precomputed once inside
`DistMultPredictor.__init__`** and cached in `self.sim_all_etypes`.

**Step 2 — metric learning.** For a queried disease, take its top-$k$ (`proto_num`) most similar
diseases and build a similarity-weighted average of *their* GNN embeddings:

$$\mathbf{h}^{\text{sim}}_i = \sum_{j \in \mathcal{D}_{\text{sim}}} \frac{\text{sim}(i,j)}{\sum_k \text{sim}(i,k)}\,\mathbf{h}_j$$

Note the `if src_h.shape[0] == src_h_keys.shape[0]` branch in `DistMultPredictor.forward`: during
**training**, query and key disease sets are identical, so `topk(k+1)[:, 1:]` drops the disease's own
self-similarity. During **evaluation** the sets differ, so it takes the plain `topk(k)`.

**Step 3 — degree-based gating.** How much to trust the auxiliary embedding is decided by node degree,
not by a learned attention (the paper found learned gates over-trust well-connected diseases):

$$c_i = 0.7\,e^{-0.7\,|\mathcal{N}^r_i|} + 0.2
\qquad
\widehat{\mathbf{h}}_i = c_i\,\mathbf{h}^{\text{sim}}_i + (1-c_i)\,\mathbf{h}_i$$

`utils.exponential(x, lamb)` with `exp_lambda=0.7`, selected by `agg_measure='rarity'`. Rare diseases
(degree 0–1) get $c \approx 0.9$; well-studied ones get $c \to 0.2$. Other `agg_measure` options
(`learn`, `avg`, `heuristics-0.8`, `100proto`) exist as ablations.

Crucially the augmented embedding is written into `h['disease']` **only for the duration of scoring one
relation**, then restored:

```python
h['disease'][h_disease['disease_query_id']] = proto_emb
out = self._apply_edges(graph, etype, h)
...
h[src][src_rel_idx] = src_h      # restore
```

### 2.4 Two-phase training

| | Pretraining | Fine-tuning |
|---|---|---|
| Edges scored | **all 29 relations** | only the 6 drug–disease relations (`dd_etypes`) |
| Batching | mini-batch over edges | **full graph**, one step per epoch |
| Disease augmentation | **off** (`pretrain_mode=True`) | **on** |
| Decoder $\mathbf{w}_r$ | trained | **re-initialised** (`xavier_uniform` in `finetune()`) |
| Loss | BCE on pos vs. sampled neg | BCE on pos vs. sampled neg |
| Typical schedule | 1–2 epochs, lr 1e-3, batch 1024 | ~500 epochs, lr 5e-4 |

The $\mathbf{w}_r$ reset is stated explicitly in the paper: it mitigates negative knowledge transfer
from generic KG relations into the clinical ones. Only the *encoder* weights carry over.

**Negative sampling.** `fix_dst` corrupts the destination node, sampling uniformly among destinations
that have in-degree > 0 for that relation (`utils.Full_Graph_NegSampler` / `Minibatch_NegSampler`).
Sampling only from "plausible" tails avoids the trivial negatives that inflate AUROC.

### 2.5 GraphMask explainability

A post-hoc, *learned* edge-sparsification: for each relation and layer a small gate network
$g(\mathbf{m}_{r,i}, \mathbf{m}_{r,j}) \in \{0,1\}$ (hard-concrete relaxation, `graphmask/`) decides
whether a message survives; masked messages are replaced by a learned baseline vector. The objective is
a Lagrangian: **minimise the number of retained edges subject to the prediction staying within
`allowance` of the unmasked prediction.** Layers are trained one at a time, deepest first
(`for layer in reversed(range(count_layers()))`).

---
## 3 · What changed versus the DGL original

The port targets **behavioural parity**, not a rewrite. Full detail in `PYG_REFACTOR.md`; the parts
that matter while reading the code:

| DGL | PyG here |
|---|---|
| `dgl.DGLHeteroGraph` | `torch_geometric.data.HeteroData` |
| `g.canonical_etypes` / `g.ntypes` | `G.edge_types` / `G.node_types` |
| `g.edges(etype=e)` → `(src, dst)` | `G[e].edge_index` → `[2, E]` |
| `g.nodes[nt].data['inp']` | `G[nt].inp` |
| `g.in_degrees(etype=e)` | `degree(G[e].edge_index[1], num_nodes=...)` |
| `g.multi_update_all(fn.mean, 'sum')` | `scatter(..., reduce='mean')` then `+=` across etypes |
| `dgl.ops.edge_softmax` | `torch_geometric.utils.softmax(e, dst_nodes, num_nodes)` |
| `graph.apply_edges(udf)` | direct indexing `h[src][ei[0]] * w_r * h[dst][ei[1]]` |
| `g.local_scope()` | removed — intermediates stay in local Python dicts |
| `EdgeDataLoader` + `MultiLayerFullNeighborSampler` | plain `torch.utils.data.DataLoader` over an edge list |

⚠️ **One design difference worth internalising.** `PYG_REFACTOR.md` proposed `LinkNeighborLoader`, but
the shipped `TxGNN.pretrain()` does something simpler: it mini-batches **only the edges being scored**,
while message passing still runs over the **entire graph** every step:

```python
# TxGNN.forward_minibatch — G is the FULL graph, not a sampled block
h_dict = self.layer1(G, input_dict)
h      = self.layer2(G, h_dict)
scores, out_pos = self.pred(pos_G, G, h, pretrain_mode, ...)   # pos_G = the batch
```

Numerically this is *equivalent to* DGL's `MultiLayerFullNeighborSampler(2)` (which also pulls in all
neighbours), but computationally it is far more expensive: you pay a full-graph 2-layer forward pass per
mini-batch. Budget accordingly — see §9.

---
## 4 · Setup

Install the PyG stack first (matching your CUDA/torch build):

```bash
pip install -r pyg_implementation/requirements.txt
# or, for wheels matching your torch build:
#   pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}+${CUDA}.html
```

In [ ]:
import os, sys, json, pickle, time

import numpy as np
import pandas as pd
import torch

REPO_ROOT = os.path.abspath(os.getcwd())
PYG_ROOT  = os.path.join(REPO_ROOT, 'pyg_implementation')

# Put the PyG implementation ahead of anything else on the path, so `import txgnn`
# resolves to pyg_implementation/txgnn and NOT to dgl_implementation/txgnn.
sys.path.insert(0, PYG_ROOT)

import txgnn
from txgnn import TxData, TxGNN, TxEval

print('txgnn package :', txgnn.__file__)
assert 'pyg_implementation' in txgnn.__file__, 'Wrong txgnn on sys.path — restart the kernel.'

In [ ]:
import torch_geometric, torch_scatter

print('torch          :', torch.__version__)
print('torch_geometric:', torch_geometric.__version__)
print('torch_scatter  :', torch_scatter.__version__)
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device    :', torch.cuda.get_device_name(0))

### 4.1 Configuration

Two profiles below. `QUICK = True` gives a smoke test that finishes in minutes; `QUICK = False`
reproduces the schedule in `pyg_implementation/reproduce/train.py`, which is what the paper numbers use.

In [ ]:
QUICK = True          # ← flip to False for a paper-scale run

# ── data ──────────────────────────────────────────────────────────────────
DATA_FOLDER = os.path.join(REPO_ROOT, 'data')   # kg.csv / node.csv / edges.csv are downloaded here
SPLIT       = 'complex_disease'                 # see §5.1 for the full list of splits
SEED        = 42

# ── model ─────────────────────────────────────────────────────────────────
N_HID, N_INP, N_OUT = 100, 100, 100   # paper / reproduce/train.py use 100
PROTO       = True                    # disease-similarity metric learning (§2.3). False ⇒ plain R-GCN
PROTO_NUM   = 3                       # k similar diseases to aggregate
ATTENTION   = False                   # True ⇒ TxGAT ablation (see the caveat in §9)
SIM_MEASURE = 'all_nodes_profile'     # disease signature vector
AGG_MEASURE = 'rarity'                # degree-based gate, c_i = 0.7*exp(-0.7*deg) + 0.2
EXP_LAMBDA  = 0.7

# ── optimisation ──────────────────────────────────────────────────────────
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

if QUICK:
    PRETRAIN_EPOCHS, PRETRAIN_LR, BATCH_SIZE = 1,   1e-3, 1024
    FINETUNE_EPOCHS, FINETUNE_LR             = 20,  5e-4
    VALID_PER_N, TRAIN_PRINT_PER_N           = 10,  5
else:
    PRETRAIN_EPOCHS, PRETRAIN_LR, BATCH_SIZE = 2,   1e-3, 1024
    FINETUNE_EPOCHS, FINETUNE_LR             = 500, 5e-4
    VALID_PER_N, TRAIN_PRINT_PER_N           = 20,  5

RUN_NAME  = f'TxGNN_pyg_{SPLIT}_seed{SEED}'
CKPT_DIR  = os.path.join(REPO_ROOT, 'saved_models', RUN_NAME)

# The six drug–disease relations that fine-tuning and evaluation optimise.
DD_ETYPES = [
    ('drug',    'contraindication',     'disease'),
    ('drug',    'indication',           'disease'),
    ('drug',    'off-label use',        'disease'),
    ('disease', 'rev_contraindication', 'drug'),
    ('disease', 'rev_indication',       'drug'),
    ('disease', 'rev_off-label use',    'drug'),
]

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'device={DEVICE}  split={SPLIT}  quick={QUICK}  ckpt={CKPT_DIR}')

---
## 5 · Stage 1 — Data and splits

`TxData.__init__` downloads three files from Harvard Dataverse into `DATA_FOLDER` (skipped if they
already exist):

| File | Content | Size |
|---|---|---|
| `kg.csv` | the raw knowledge graph with human-readable names | ~1 GB |
| `node.csv` | node index → id/type/name | small |
| `edges.csv` | edge list | large |

`prepare_split()` then does the real work:

1. **`preprocess_kg`** converts the undirected KG into a *directed* `kg_directed.csv`, adding a
   `rev_<rel>` edge for every relation and re-indexing nodes **per node type** (so `x_idx` is a
   0-based index within `drug`, `disease`, …). First run takes several minutes; cached afterwards.
2. **`create_split`** produces `train.csv` / `valid.csv` / `test.csv` under
   `<DATA_FOLDER>/<split>_<seed>/`, again cached.
3. **`create_pyg_graph(df_train, df)`** builds the `HeteroData`. Note it takes edges from
   `df_train` but node counts from the **full** `df`, so held-out diseases still exist as nodes with no
   treatment edges — that's what makes the zero-shot setting well-defined.

### 5.1 Which split to use

| `split` | What is held out | Use for |
|---|---|---|
| `random` | random 5% of drug–disease edges | easy baseline; over-optimistic |
| `complex_disease` | diseases chosen so *all* their treatment edges move to test | **the main zero-shot benchmark** |
| `complex_disease_cv` | same, cross-validation folds | variance estimates |
| `cell_proliferation`, `mental_health`, `cardiovascular`, `anemia`, `adrenal_gland`, `autoimmune`, `metabolic_disorder`, `diabetes`, `neurodigenerative` | an entire **disease area** (holds out that area *and* removes its one-hop KG edges) | the "unseen disease area" experiments in Fig. 2 |
| `disease_eval` | a single disease (`disease_eval_idx`) | per-disease case studies |
| `full_graph` | nothing | train a deployment model; evaluation becomes prediction-only |
| `no_kg=True` | keeps only the 3 clinical relations | KG-ablation |

Disease-area splits are the hardest and the ones the zero-shot claims rest on.

In [ ]:
t0 = time.time()

data = TxData(data_folder_path=DATA_FOLDER)     # downloads kg.csv / node.csv / edges.csv if missing
data.prepare_split(split=SPLIT, seed=SEED, no_kg=False)

print(f'\nprepare_split took {time.time() - t0:.1f}s')
print('train / valid / test edges:',
      len(data.df_train), len(data.df_valid), len(data.df_test))

### 5.2 Inspect the graph

`data.G` is a `HeteroData`. Everything downstream indexes it by node type string or by the canonical
3-tuple `(src_type, relation, dst_type)`.

In [ ]:
G = data.G

print(f'{len(G.node_types)} node types, {len(G.edge_types)} edge types, '
      f'{sum(G[et].num_edges for et in G.edge_types):,} directed edges\n')

nodes_df = pd.DataFrame(
    [(nt, G[nt].num_nodes) for nt in G.node_types],
    columns=['node_type', 'num_nodes'],
).sort_values('num_nodes', ascending=False)
display(nodes_df)

In [ ]:
edges_df = pd.DataFrame(
    [(s, r, d, G[(s, r, d)].num_edges) for (s, r, d) in G.edge_types],
    columns=['src_type', 'relation', 'dst_type', 'num_edges'],
).sort_values('num_edges', ascending=False)

print('--- the 6 drug–disease relations we actually optimise ---')
display(edges_df[edges_df.apply(lambda r: (r.src_type, r.relation, r.dst_type) in DD_ETYPES, axis=1)])

print('--- top 15 relations by edge count ---')
display(edges_df.head(15))

In [ ]:
# How sparse is the treatment signal? This is the motivation for §2.3.
from torch_geometric.utils import degree

et = ('disease', 'rev_indication', 'drug')
deg = degree(G[et].edge_index[0], num_nodes=G['disease'].num_nodes)

n_disease   = G['disease'].num_nodes
n_with_drug = int((deg > 0).sum())
print(f'diseases in the KG                       : {n_disease:,}')
print(f'…with ≥1 indication edge in train        : {n_with_drug:,} ({100*n_with_drug/n_disease:.1f}%)')
print(f'median indications among those           : {deg[deg > 0].median():.0f}')
print(f'→ the gate c_i = 0.7*exp(-0.7*deg)+0.2 is ~{0.7*np.exp(-0.7*0)+0.2:.2f} at degree 0, '
      f'{0.7*np.exp(-0.7*10)+0.2:.2f} at degree 10')

---
## 6 · Stage 2 — Build the model

`model_initialize()` is doing more than allocating layers. In order:

1. Moves `G` to **CPU** (graph construction below is CPU-only).
2. `initialize_node_embedding(G, n_inp)` — attaches a frozen Xavier-uniform `G[nt].inp` per node type.
   These are inputs, *not* parameters; only the relation weight matrices, `w_rels`, and (if `proto`)
   the gate linears are learned.
3. `evaluate_graph_construct(df_valid, …)` and the same for `df_test` — builds the positive eval graphs
   plus a matched `fix_dst` negative graph each. **These negatives are sampled once and frozen**, so
   validation/test numbers are comparable across epochs.
4. Constructs `HeteroRGCN`, which in turn constructs `DistMultPredictor`. If `proto=True`, the
   constructor **precomputes the full disease×disease similarity matrix per drug–disease relation**.
   This is the slow, memory-hungry part of initialisation (see §9).

In [ ]:
model = TxGNN(
    data              = data,
    weight_bias_track = False,     # True ⇒ log to Weights & Biases (needs `pip install wandb`)
    proj_name         = 'TxGNN',
    exp_name          = RUN_NAME,
    device            = DEVICE,
)

t0 = time.time()
model.model_initialize(
    n_hid       = N_HID,
    n_inp       = N_INP,
    n_out       = N_OUT,
    proto       = PROTO,
    proto_num   = PROTO_NUM,
    attention   = ATTENTION,
    sim_measure = SIM_MEASURE,
    agg_measure = AGG_MEASURE,
    exp_lambda  = EXP_LAMBDA,
)
print(f'model_initialize took {time.time() - t0:.1f}s')

In [ ]:
from txgnn.utils import get_n_params

print(f'trainable parameters: {get_n_params(model.model):,}\n')

groups = {}
for name, p in model.model.named_parameters():
    groups.setdefault(name.split('.')[0], 0)
    groups[name.split('.')[0]] += p.numel()
for k, v in sorted(groups.items(), key=lambda kv: -kv[1]):
    print(f'  {k:<12} {v:>12,}')

# Frozen inputs are on the graph, not in named_parameters():
inp_params = sum(G[nt].inp.numel() for nt in G.node_types)
print(f'\nfrozen input embeddings (on G, requires_grad=False): {inp_params:,}')

# Cached similarity matrices, if proto=True
if PROTO:
    for et, sim in model.model.pred.sim_all_etypes.items():
        print(f'  sim matrix {str(et):<45} {tuple(sim.shape)}  '
              f'{sim.element_size()*sim.nelement()/1e9:.2f} GB')

---
## 7 · Stage 3 — Pretraining on all relations

**Goal:** push generic biomedical knowledge (protein interactions, pathways, phenotypes, …) into the
encoder weights before specialising on treatment prediction.

What `TxGNN.pretrain()` does per step:

1. Take a mini-batch of `batch_size` edges sampled uniformly across **all 29 relations**.
2. Assemble them into a positive `HeteroData` (`pos_g`) and draw matched negatives with
   `Minibatch_NegSampler(..., 'fix_dst')`.
3. `forward_minibatch` runs both encoder layers over the **full graph**, then scores `pos_g` and
   `neg_g` with DistMult and a sigmoid — with `pretrain_mode=True`, so the disease-similarity
   augmentation is bypassed entirely.
4. BCE loss, AdamW, no LR schedule.

Skip this stage entirely when `no_kg=True` (it raises), and note that only the **encoder** survives into
fine-tuning — `w_rels` is reset.

In [ ]:
if data.no_kg:
    print('no_kg split — pretraining is skipped (it would be identical to fine-tuning).')
else:
    t0 = time.time()
    model.pretrain(
        n_epoch           = PRETRAIN_EPOCHS,
        learning_rate     = PRETRAIN_LR,
        batch_size        = BATCH_SIZE,
        train_print_per_n = 20,
    )
    print(f'\npretrain took {(time.time() - t0)/60:.1f} min')

---
## 8 · Stage 4 — Fine-tuning on drug–disease relations

Now the loss is restricted to the six `dd_etypes`, the disease-similarity module is **on**, and training
is **full-graph**: one optimiser step per epoch over every drug–disease edge at once. That is why
`n_epoch=500` is cheap in wall-clock terms relative to pretraining.

Per epoch:

1. `Full_Graph_NegSampler(..., 'fix_dst')` redraws negatives — **fresh every epoch**, unlike the frozen
   validation negatives.
2. Full forward: 2-layer encoder → disease augmentation → DistMult → sigmoid → BCE.
3. `AdamW` + `ReduceLROnPlateau(mode='min', factor=0.8)` on the training loss.
4. Every `valid_per_n` epochs, `evaluate_fb` on the frozen validation graphs. **Model selection is on
   validation macro-AUROC** — `self.best_model` is a deepcopy of the best checkpoint, and it is
   `best_model` (not `model`) that gets tested and saved.

The very first line of `finetune()` re-initialises the decoder:

```python
torch.nn.init.xavier_uniform(self.model.w_rels)   # discard pretrained w_r
```

At the end it automatically runs the **test** evaluation and prints per-relation AUROC/AUPRC.

In [ ]:
t0 = time.time()
model.finetune(
    n_epoch           = FINETUNE_EPOCHS,
    learning_rate     = FINETUNE_LR,
    train_print_per_n = TRAIN_PRINT_PER_N,
    valid_per_n       = VALID_PER_N,
    save_name         = None,      # or a path to pickle the test-metric dict
)
print(f'\nfinetune took {(time.time() - t0)/60:.1f} min')

### 8.1 Save / reload

`save_model` writes `config.pkl` (the `model_initialize` kwargs) + `model.pt` (`best_model` state dict).
`load_pretrained` replays `model_initialize(**config)` and then loads the weights — so reloading needs
the *same* `TxData` split, and it re-runs the similarity precomputation.

In [ ]:
os.makedirs(CKPT_DIR, exist_ok=True)
model.save_model(CKPT_DIR)
print('saved →', CKPT_DIR, os.listdir(CKPT_DIR))

In [ ]:
# ── Reload path (fresh session) ───────────────────────────────────────────
# data  = TxData(data_folder_path=DATA_FOLDER)
# data.prepare_split(split=SPLIT, seed=SEED, no_kg=False)
# model = TxGNN(data=data, device=DEVICE)
# model.load_pretrained(CKPT_DIR)

---
## 9 · Stage 5 — Evaluation

Two complementary views.

**(a) Edge-level, `evaluate_fb`.** Pools all positive test edges against the frozen negatives and reports
micro/macro AUROC and AUPRC. Good for tracking, but it flatters the model: negatives are sampled, so the
ratio is 1:1 rather than the real 1:(#drugs).

**(b) Disease-centric, `TxEval.eval_disease_centric`.** The evaluation the paper actually reports. For
each held-out disease it scores **every drug in the KG**, ranks them, and computes
`AUROC`, `AUPRC`, `Recall@{1%,5%,10%}`, `Recall@{10,50,100}`, `MRR@{10,50,100}`, `AP@{10,50,100}`.
This is the realistic retrieval setting: one disease, thousands of candidate drugs, a handful of true
indications.

In [ ]:
from txgnn.utils import evaluate_fb

G_dev = data.G.to(DEVICE)
(auroc_rel, auprc_rel, micro_auroc, micro_auprc, macro_auroc, macro_auprc), loss = evaluate_fb(
    model.best_model, model.g_test_pos, model.g_test_neg, G_dev, DD_ETYPES, DEVICE, mode='test'
)

print(f'test loss        {loss:.4f}')
print(f'micro AUROC/AUPRC {micro_auroc:.4f} / {micro_auprc:.4f}')
print(f'macro AUROC/AUPRC {macro_auroc:.4f} / {macro_auprc:.4f}\n')

display(pd.DataFrame({
    'AUROC': {str(k): v for k, v in auroc_rel.items()},
    'AUPRC': {str(k): v for k, v in auprc_rel.items()},
}).loc[[str(e) for e in DD_ETYPES if str(e) in {str(k) for k in auroc_rel}]])

In [ ]:
evaluator = TxEval(model=model)

# Diseases whose `indication` edges landed in the test set — the zero-shot targets.
test_diseases = evaluator.retrieve_disease_idxs_test_set('indication')
print(f'{len(test_diseases)} test diseases for `indication`')

result = evaluator.eval_disease_centric(
    disease_idxs   = 'test_set',   # or a list like [12661.0, 11318.0] for specific diseases
    relation       = 'indication',
    show_plot      = False,
    verbose        = True,
    return_raw     = False,
    simulate_random= True,         # also reports a random-ranking baseline for context
    save_result    = True,
    save_name      = os.path.join(CKPT_DIR, 'disease_centric_eval.pkl'),
)

In [ ]:
# `result` maps metric name → {disease_idx: value}. Summarise across diseases.
metric_keys = ['AUROC', 'AUPRC', 'Recall@1%', 'Recall@5%', 'Recall@10%',
               'Recall@10', 'Recall@50', 'Recall@100',
               'MRR@10', 'MRR@50', 'MRR@100', 'AP@10', 'AP@50', 'AP@100']

rows = []
for k in metric_keys:
    if k in result and isinstance(result[k], dict) and len(result[k]):
        vals = np.array(list(result[k].values()), dtype=float)
        rows.append({'metric': k, 'mean': vals.mean(), 'std': vals.std(), 'n_diseases': len(vals)})

display(pd.DataFrame(rows).set_index('metric').round(4))

---
## 10 · Stage 6 — Using the trained model

Three entry points:

- **`predict(df)`** — score arbitrary `(x_idx, relation, y_idx)` triples. Indices are the *per-node-type*
  indices from `kg_directed.csv`; use `data.retrieve_id_mapping()` to translate to/from names.
- **`retrieve_embedding()`** — the post-encoder node embeddings $\mathbf{h}$, per node type.
- **`retrieve_sim_diseases(relation, k)`** — for each disease, the $k$ nearest diseases under the
  signature-vector similarity. This is the interpretable half of §2.3: it tells you *which* diseases the
  model borrowed evidence from.

In [ ]:
mapping = data.retrieve_id_mapping()
print(mapping.keys())

idx2name_disease = {idx: mapping['id2name_disease'].get(i, i)
                    for idx, i in mapping['idx2id_disease'].items()}
idx2name_drug    = {idx: mapping['id2name_drug'].get(i, i)
                    for idx, i in mapping['idx2id_drug'].items()}

print('\nexample diseases:', list(idx2name_disease.items())[:5])
print('example drugs   :', list(idx2name_drug.items())[:5])

In [ ]:
# Score every drug for one held-out disease and show the top candidates.
#
# `predict()` always scores all six clinical relations in one pass, so the query frame must contain
# rows for each of them: a relation with zero rows becomes an empty edge set, and the `proto` branch
# of DistMultPredictor cannot build a similarity block for an empty query. Note the orientation —
# `indication` is drug→disease, `rev_indication` is disease→drug.
disease_idx = float(test_diseases[0])
drug_idxs   = np.arange(data.G['drug'].num_nodes, dtype=float)
print('disease:', idx2name_disease.get(disease_idx, disease_idx))

frames = []
for rel in ['indication', 'contraindication', 'off-label use']:
    frames.append(pd.DataFrame({'x_idx': drug_idxs,   'relation': rel,          'y_idx': disease_idx}))
    frames.append(pd.DataFrame({'x_idx': disease_idx, 'relation': 'rev_' + rel, 'y_idx': drug_idxs}))
df_query = pd.concat(frames, ignore_index=True)

# predict() returns raw DistMult logits from `self.model` (the last-epoch model, NOT `best_model`);
# apply the sigmoid yourself to get p_{i,j,r}.
scores = model.predict(df_query)
s = torch.sigmoid(scores[('disease', 'rev_indication', 'drug')]).detach().cpu().numpy()

top = np.argsort(-s)[:15]
display(pd.DataFrame({
    'drug' : [idx2name_drug.get(float(i), i) for i in top],
    'p'    : s[top],
}))

In [ ]:
# Node embeddings (dict: node_type -> Tensor[num_nodes, n_out])
h = model.retrieve_embedding()
print({k: tuple(v.shape) for k, v in h.items()})

# Which diseases did the model treat as neighbours?
sim_diseases = model.retrieve_sim_diseases(relation='indication', k=5)
print('\nsimilar-disease index tensor:', tuple(sim_diseases.shape))

---
## 11 · Stage 7 — Explainability with GraphMask *(optional)*

Trains gates on top of the **already fine-tuned** model (`best_model` is deep-copied; all original
weights are frozen via `disable_all_gradients`). Layers are unlocked one at a time, deepest first.

Knobs:

| Arg | Meaning |
|---|---|
| `relation` | which clinical relation to explain: `indication` / `contraindication` / `off-label` |
| `allowance` | how far masked predictions may drift from the originals before the constraint bites |
| `penalty_scaling` | weight on the sparsity term (higher ⇒ fewer edges kept) |
| `epochs_per_layer` | Lagrangian steps per layer — the paper uses ~1000, so this is a long job |
| `no_base` | drop the learned baseline vector; masked messages become exactly zero |

The printed `num_masked_l1 / num_masked_l2` are the fractions of edges masked at each layer — that is
your sparsity readout.

In [ ]:
RUN_GRAPHMASK = False     # ← this is a long job; opt in explicitly

if RUN_GRAPHMASK:
    metrics = model.train_graphmask(
        relation                  = 'indication',
        learning_rate             = 3e-4,
        allowance                 = 0.005,
        epochs_per_layer          = 1000 if not QUICK else 5,
        penalty_scaling           = 1,
        moving_average_window_size= 100,
        valid_per_n               = 20,
    )
    model.save_graphmask_model(os.path.join(CKPT_DIR, 'graphmask'))
    print(metrics)

In [ ]:
if RUN_GRAPHMASK:
    # Per-edge gates / scores / penalties, for the whole graph and for the test graph.
    whole_graph, test_graph = model.retrieve_gates_scores_penalties(relation='indication')
    (orig_pos, orig_neg, upd_pos, upd_neg, num_masked, gates, scores, penalties) = whole_graph

    total_edges = sum(model.G[et].num_edges for et in model.G.edge_types)
    print(f'layer 1 masked: {num_masked[0]/total_edges:.2%}')
    print(f'layer 2 masked: {num_masked[1]/total_edges:.2%}')

    # NOTE: `model.retrieve_save_gates(path)` builds a named edge table from these scores, but it is
    # currently broken on pandas ≥ 2.0 (uses the removed DataFrame.append) and calls
    # retrieve_gates_scores_penalties() without the required `relation` argument. Build the table
    # manually from `scores` if you need it — see §12.

---
## 12 · Known rough edges (read before a long run)

These are real properties of the current code, verified by reading it — not hypotheticals.

**Cost / memory**

1. **`pretrain()` materialises every edge as a Python tuple.** `TxGNN.py` builds
   `all_edges = [(etype, src, dst) for every edge]` with a Python loop over `edge_index` columns. On the
   full ~8 M-edge KG that is a multi-GB list and takes a long time before the first optimiser step.
   Vectorise it (build a per-relation offset table and index with tensors) if you are running at full
   scale.
2. **Mini-batching does not reduce the encoder cost.** Every pretraining step still runs both encoder
   layers over the *entire* graph (§3). Steps-per-epoch scales as `num_edges / batch_size`, and each step
   is a full-graph forward. Larger `batch_size` is close to free here.
3. **`proto=True` precomputes dense disease×disease similarity matrices**, one per drug–disease relation
   that exists in the graph. With ~17 K diseases that is ~1.2 GB in fp32 *per relation*, held for the
   lifetime of the model. Set `proto=False` to sanity-check the pipeline on a small machine.
4. **Fine-tuning is full-graph** — the whole KG and all embeddings must fit on the device.

**Correctness caveats**

5. **`attention=True` is incompatible with `pretrain()`.** `forward_minibatch` calls
   `self.layer1(G, input_dict)` without the `return_att` argument, and `AttHeteroRGCNLayer.forward`
   returns a `(out, att)` tuple regardless — so the TxGAT variant works with `finetune()` but crashes in
   pretraining. `reproduce/train.py` sidesteps this by only pretraining when `model == 'TxGNN'`.
6. **`retrieve_save_gates(path)` is broken twice over**: it calls
   `self.retrieve_gates_scores_penalties()` without its required `relation` argument, and it accumulates
   with `DataFrame.append`, removed in pandas 2.0. Use `retrieve_gates_scores_penalties(relation=...)`
   and `pd.concat` yourself.
7. **`finetune()` calls the deprecated `torch.nn.init.xavier_uniform`** (no trailing underscore). It
   still works, but emits a warning — and `warnings.filterwarnings("ignore")` at the top of the module
   hides it.
8. **The encoder omits the paper's residual term** $\mathbf{h}^{(l-1)}_i$ (§2.1). Intentional: the port
   matches the reference DGL code.
9. **Validation/test negatives are frozen at `model_initialize` time**; training negatives are redrawn
   every epoch. Comparing the two loss curves directly is misleading.
10. **`predict()` scores `self.model`, not `self.best_model`** — i.e. the last fine-tuning epoch, while
    `evaluate_fb`/`TxEval` use the validation-selected `best_model`. Assign
    `model.model = model.best_model` first if you want inference to match the reported metrics.

**Practical**

11. `model_initialize()` moves `G` to CPU and `finetune()` moves it back to `DEVICE`. If you call
    low-level utilities between the two, check `G[nt].inp.device` first.
12. Splits and the processed KG are cached on disk by `(split, seed)`. Change `SEED` and you pay the
    several-minute `create_split` cost again — but you also get a genuinely different split.
13. For multi-seed runs, `pyg_implementation/reproduce/run_txgnn.sh` + `train.py` are the batch entry
    point; this notebook is the interactive equivalent of `train.py`.

---

## 13 · Where to go next

- **Parity with the DGL original** — [`pyg_benchmark.ipynb`](pyg_benchmark.ipynb) and
  [`dgl_benchmark.ipynb`](dgl_benchmark.ipynb) run the same stages in both frameworks and write to
  `comparison_outputs/`; [`compare_results.ipynb`](compare_results.ipynb) diffs them.
- **Batch experiments** — `pyg_implementation/reproduce/train.py`.
- **Method details** — `txgnn_article/main.md`, Methods section (`#### Methods` onwards).
- **Port design rationale** — `PYG_REFACTOR.md`.